# database check

In [3]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "institutional_holding.db"

connection = sqlite3.connect(DB_PATH)
print(f"数据库位置：{DB_PATH}")
print(f"数据库存在：{DB_PATH.exists()}")

数据库位置：/home/ubuntu/institutional-holding-tracker/data/institutional_holding.db
数据库存在：True


## 查看数据库表

In [5]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

tables

,name
0,alerts
1,daily_prices
2,fund_holdings
3,holder_mappings
4,holding_changes_summary
5,index_components
6,index_holding_summary
7,indices
8,institutional_holdings
9,institutional_research


## 查看指数基本信息

In [7]:
indices = pd.read_sql_query(
    "SELECT * FROM indices ORDER BY index_code",
    connection,
)

indices

,id,index_name,index_code,exchange,component_count,updated_at
0,25,沪深300,000300,sh,300,2026-08-23 03:49:36
1,28,科创50,000688,sh,50,2026-08-23 03:49:36
2,26,中证500,000905,sh,500,2026-08-23 03:49:36
3,27,创业板指,399006,sz,100,2026-08-23 03:49:36


## 查看股票数量

In [18]:
stocks = pd.read_sql_query(
    "SELECT COUNT(*) AS stock_count FROM stocks",
    connection,
)

stocks


,stock_count
0,1694


## 查看指数成分股

In [19]:
components = pd.read_sql_query(
    """
    SELECT index_code, stock_code, stock_name, weight, effective_date
    FROM index_components
    ORDER BY index_code, stock_code
    LIMIT 20
    """,
    connection,
)

components

,index_code,stock_code,stock_name,weight,effective_date
0,000300,000001,平安银行,0.433,2026-08-23
1,000300,000002,万科A,0.087,2026-08-23
2,000300,000063,中兴通讯,0.418,2026-08-23
3,000300,000100,TCL科技,0.378,2026-08-23
4,000300,000157,中联重科,0.145,2026-08-23
5,000300,000166,申万宏源,0.162,2026-08-23
6,000300,000301,东方盛虹,0.119,2026-08-23
7,000300,000333,美的集团,1.634,2026-08-23
8,000300,000338,潍柴动力,0.577,2026-08-23
9,000300,000408,藏格矿业,0.243,2026-08-23


In [8]:
component_counts = pd.read_sql_query(
    """
    SELECT index_code, COUNT(DISTINCT stock_code) AS stock_count
    FROM index_components
    GROUP BY index_code
    ORDER BY index_code
    """,
    connection,
)

component_counts

,index_code,stock_count
0,000300,300
1,000688,50
2,000905,500
3,399006,100


## 统计所有指数并随机查看成分股

In [9]:
all_components = pd.read_sql_query(
    """
    SELECT index_code, stock_code, stock_name, weight, effective_date
    FROM index_components
    ORDER BY index_code, stock_code
    """,
    connection,
)

index_summary = (
    all_components.groupby("index_code", as_index=False)
    .agg(stock_count=("stock_code", "nunique"))
    .sort_values("index_code")
)

print(f"实际写入的指数数量：{len(index_summary)}")
index_summary

实际写入的指数数量：4


,index_code,stock_count
0,000300,300
1,000688,50
2,000905,500
3,399006,100


In [7]:
random_samples = pd.concat(
    [
        group.sample(n=min(5, len(group)), random_state=42)
        for _, group in all_components.groupby("index_code")
    ],
    ignore_index=True,
).sort_values(["index_code", "stock_code"])

random_samples

,index_code,stock_code,stock_name,weight,effective_date
3,000300,000408,藏格矿业,0.243,2026-08-18
2,000300,600372,中航机载,0.105,2026-08-18
0,000300,601077,渝农商行,0.138,2026-08-18
4,000300,601633,长城汽车,0.080,2026-08-18
1,000300,603019,中科曙光,0.469,2026-08-18


## 诊断指数基本信息写入

In [10]:
from config.settings import TRACKED_INDICES
from database.db_manager import query_sql
from ingestion.index_components import _update_indices_info

print("配置中的指数：")
for name, info in TRACKED_INDICES.items():
    print(name, info)

_update_indices_info()

updated_indices = pd.DataFrame(query_sql(
    "SELECT * FROM indices ORDER BY index_code"
))
updated_indices

配置中的指数：
沪深300 {'code': '000300', 'exchange': 'sh'}
中证500 {'code': '000905', 'exchange': 'sh'}
创业板指 {'code': '399006', 'exchange': 'sz'}
科创50 {'code': '000688', 'exchange': 'sh'}


,id,index_name,index_code,exchange,component_count,updated_at
0,29,沪深300,000300,sh,300,2026-08-23 04:38:15
1,32,科创50,000688,sh,50,2026-08-23 04:38:15
2,30,中证500,000905,sh,500,2026-08-23 04:38:15
3,31,创业板指,399006,sz,100,2026-08-23 04:38:15


## 回滚本次诊断写入

In [10]:
index_codes = tuple(info["code"] for info in TRACKED_INDICES.values())
placeholders = ", ".join("?" for _ in index_codes)

connection.execute(
    f"DELETE FROM indices WHERE index_code IN ({placeholders})",
    index_codes,
)
connection.commit()

remaining_indices = pd.read_sql_query(
    "SELECT * FROM indices ORDER BY index_code",
    connection,
)
remaining_components = pd.read_sql_query(
    """
    SELECT index_code, COUNT(DISTINCT stock_code) AS stock_count
    FROM index_components
    GROUP BY index_code
    ORDER BY index_code
    """,
    connection,
)

print(f"回滚后 indices 记录数：{len(remaining_indices)}")
print("成分股明细统计：")
remaining_components

回滚后 indices 记录数：0
成分股明细统计：


,index_code,stock_count
0,000300,300


# 小样本采集测试

## 数据表

In [11]:
table_check = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table' AND name = 'top_holders'
    """,
    connection,
)

table_check

,name
0,top_holders


## 单只股票测试接口

In [19]:
from ingestion.top_holders import fetch_top_holders_em

df = fetch_top_holders_em("000001", "20250331", tRUE)

print("shape:", df.shape)
print("columns:", list(df.columns))
display(df.head(10))

shape: (10, 7)
columns: ['名次', '股东名称', '股份类型', '持股数', '占总股本持股比例', '增减', '变动比率']


,名次,股东名称,股份类型,持股数,占总股本持股比例,增减,变动比率
0,1,中国平安保险(集团)股份有限公司-集团本级-自有资金,流通A股,9618540236,49.56,不变,NaN
1,2,中国平安人寿保险股份有限公司-自有资金,流通A股,1186100488,6.11,不变,NaN
2,3,香港中央结算有限公司,流通A股,658114653,3.39,-88767070,-11.885024
3,4,中国平安人寿保险股份有限公司-传统-普通保险产品,流通A股,440478714,2.27,不变,NaN
4,5,中国证券金融股份有限公司,流通A股,429232688,2.21,不变,NaN
5,6,中国工商银行股份有限公司-华泰柏瑞沪深300交易型开放式指数证券投资基金,流通A股,158803503,0.82,-8714000,-5.201844
6,7,中国建设银行股份有限公司-易方达沪深300交易型开放式指数发起式证券投资基金,流通A股,110935144,0.57,-4615700,-3.994519
7,8,中国工商银行股份有限公司-华夏沪深300交易型开放式指数证券投资基金,流通A股,75280777,0.39,-1530300,-1.992291
8,9,中国银行股份有限公司-嘉实沪深300交易型开放式指数证券投资基金,流通A股,70005962,0.36,-2766300,-3.801311
9,10,深圳中电投资有限公司,流通A股,62523366,0.32,不变,NaN


In [21]:
from ingestion.top_holders import ingest_all_top_holders

ingest_all_top_holders(
    ["000001"],
    ["20250331", "20241231"],
)

2026-08-19 19:52:41 [INFO] ingestion.top_holders: [TopHolders] Start ingesting 十大股东 for 1 stocks x 2 dates...
2026-08-19 19:52:43 [INFO] ingestion.top_holders: [TopHolders] Ingestion completed. Total records: 20
2026-08-19 19:52:43 [INFO] ingestion.top_holders: [TopHolders] Start ingesting 十大流通股东 for 1 stocks x 2 dates...
2026-08-19 19:52:44 [INFO] ingestion.top_holders: [TopHolders] Ingestion completed. Total records: 20


In [24]:
# 获取沪深300（000300）300只成分股的十大股东数据
top_holders_schema = pd.read_sql_query(
    "PRAGMA table_info(top_holders)",
    connection,
)

display(top_holders_schema)

top_holders_50 = pd.read_sql_query(
    """
    SELECT th.*
    FROM top_holders AS th
    INNER JOIN (
        SELECT DISTINCT stock_code
        FROM index_components
        WHERE index_code = '000300'
    ) AS c
        ON c.stock_code = th.stock_code
    ORDER BY th.stock_code
    """,
    connection,
)

print(f"获取记录数：{len(top_holders_50)}")
print(f"覆盖股票数：{top_holders_50['stock_code'].nunique()}")

display(top_holders_50)

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,NaN,1
1,1,stock_code,TEXT,1,NaN,0
2,2,stock_name,TEXT,0,NaN,0
3,3,report_date,DATE,1,NaN,0
4,4,holder_name,TEXT,1,NaN,0
5,5,holder_type,TEXT,0,NaN,0
6,6,holder_type_raw,TEXT,0,NaN,0
7,7,hold_shares,REAL,0,NaN,0
8,8,hold_ratio_total,REAL,0,NaN,0
9,9,hold_ratio_float,REAL,0,NaN,0


获取记录数：1960
覆盖股票数：49


,id,stock_code,stock_name,report_date,holder_name,holder_type,holder_type_raw,hold_shares,hold_ratio_total,hold_ratio_float,change_status,change_shares,change_ratio,rank,is_float_holder,announce_date,data_source,created_at
0,5921,000001,None,2024-12-31,中国平安保险(集团)股份有限公司-集团本级-自有资金,NaN,None,9.618540e+09,49.56,NaN,不变,None,None,1,0,None,akshare,2026-08-19 11:59:50
1,5922,000001,None,2024-12-31,中国平安人寿保险股份有限公司-自有资金,NaN,None,1.186100e+09,6.11,NaN,不变,None,None,2,0,None,akshare,2026-08-19 11:59:50
2,5923,000001,None,2024-12-31,香港中央结算有限公司,NaN,None,7.468817e+08,3.85,NaN,减持,None,None,3,0,None,akshare,2026-08-19 11:59:50
3,5924,000001,None,2024-12-31,中国平安人寿保险股份有限公司-传统-普通保险产品,NaN,None,4.404787e+08,2.27,NaN,不变,None,None,4,0,None,akshare,2026-08-19 11:59:50
4,5925,000001,None,2024-12-31,中国证券金融股份有限公司,NaN,None,4.292327e+08,2.21,NaN,不变,None,None,5,0,None,akshare,2026-08-19 11:59:50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1955,5866,002304,None,2025-03-31,中国银行股份有限公司-易方达蓝筹精选混合型证券投资基金,其他,None,2.550000e+07,NaN,1.692767,减持,None,None,6,1,None,akshare,2026-08-18 12:52:55
1956,5867,002304,None,2025-03-31,香港中央结算有限公司,北向资金,None,2.315466e+07,NaN,1.537076,减持,None,None,7,1,None,akshare,2026-08-18 12:52:55
1957,5868,002304,None,2025-03-31,中国证券金融股份有限公司,证金公司,None,1.379004e+07,NaN,0.915425,不变,None,None,8,1,None,akshare,2026-08-18 12:52:55
1958,5869,002304,None,2025-03-31,邢福平,其他,None,1.029060e+07,NaN,0.683121,减持,None,None,9,1,None,akshare,2026-08-18 12:52:55


# 十大股东数据

## 验证数据库

In [33]:
# 查看十大股东数据是否已写入数据库
summary = pd.read_sql_query(
    """
    SELECT report_date, is_float_holder, COUNT(*) AS row_count,
           COUNT(DISTINCT stock_code) AS stock_count
    FROM top_holders
    GROUP BY report_date, is_float_holder
    ORDER BY report_date, is_float_holder
    """,
    connection,
)

display(summary)

print("总记录数:", pd.read_sql_query("SELECT COUNT(*) AS count FROM top_holders", connection).iloc[0, 0])

latest_records = pd.read_sql_query(
    """
    SELECT *
    FROM top_holders
    ORDER BY report_date DESC, stock_code, rank
    """,
    connection,
)

display(
    latest_records.groupby("report_date", group_keys=False)
    .head(5)
)

#display(pd.read_sql_query("SELECT * FROM top_holders LIMIT 5", connection))

,report_date,is_float_holder,row_count,stock_count
0,2024-12-31,0,16741,1668
1,2024-12-31,1,16665,1660
2,2025-03-31,0,16705,1664
3,2025-03-31,1,16687,1663
4,2025-06-30,0,16789,1674
5,2025-06-30,1,16729,1668
6,2025-09-30,0,16788,1675
7,2025-09-30,1,16773,1674
8,2025-12-31,0,16936,1690
9,2025-12-31,1,16854,1682


总记录数: 224721


,id,stock_code,stock_name,report_date,holder_name,holder_type,holder_type_raw,hold_shares,hold_ratio_total,hold_ratio_float,change_status,change_shares,change_ratio,rank,is_float_holder,announce_date,data_source,created_at
0,102294,000001,平安银行,2026-06-30,中国平安保险(集团)股份有限公司-集团本级-自有资金,其他,None,9.618540e+09,49.56,NaN,不变,None,None,1,0,None,akshare,2026-08-20 06:57:12
1,104127,000001,平安银行,2026-06-30,中国平安保险(集团)股份有限公司-集团本级-自有资金,其他,None,9.618540e+09,NaN,49.565580,不变,None,None,1,1,None,akshare,2026-08-20 07:22:09
2,102295,000001,平安银行,2026-06-30,中国平安人寿保险股份有限公司-自有资金,保险,None,1.186100e+09,6.11,NaN,不变,None,None,2,0,None,akshare,2026-08-20 06:57:12
3,104128,000001,平安银行,2026-06-30,中国平安人寿保险股份有限公司-自有资金,保险,None,1.186100e+09,NaN,6.112129,不变,None,None,2,1,None,akshare,2026-08-20 07:22:09
4,102296,000001,平安银行,2026-06-30,香港中央结算有限公司,北向资金,None,7.368784e+08,3.80,NaN,增持,None,None,3,0,None,akshare,2026-08-20 06:57:12
23185,86021,000001,平安银行,2026-03-31,中国平安保险(集团)股份有限公司-集团本级-自有资金,其他,None,9.618540e+09,49.56,NaN,不变,None,None,1,0,None,akshare,2026-08-20 06:09:10
23186,94159,000001,平安银行,2026-03-31,中国平安保险(集团)股份有限公司-集团本级-自有资金,其他,None,9.618540e+09,NaN,49.565795,不变,None,None,1,1,None,akshare,2026-08-20 06:22:51
23187,86022,000001,平安银行,2026-03-31,中国平安人寿保险股份有限公司-自有资金,保险,None,1.186100e+09,6.11,NaN,不变,None,None,2,0,None,akshare,2026-08-20 06:09:10
23188,94160,000001,平安银行,2026-03-31,中国平安人寿保险股份有限公司-自有资金,保险,None,1.186100e+09,NaN,6.112156,不变,None,None,2,1,None,akshare,2026-08-20 06:22:51
23189,86023,000001,平安银行,2026-03-31,香港中央结算有限公司,北向资金,None,5.707720e+08,2.94,NaN,减持,None,None,3,0,None,akshare,2026-08-20 06:09:10


## 个股的十大股东

In [ ]:
from database.db_manager import query_df

df = query_df("""
    SELECT rank, holder_name, holder_type, hold_shares,
           hold_ratio_total, change_status, change_ratio
    FROM top_holders
    WHERE stock_code = '601988'
      AND report_date = '2026-06-30'      -- 换成任意报告期
      AND is_float_holder = 0             -- 0=十大股东, 1=十大流通股东
    ORDER BY rank
""")
print(df)


   rank                         holder_name holder_type   hold_shares  \
0     1                        中央汇金投资有限责任公司        汇金公司  1.887919e+11   
1     2                     香港中央结算(代理人)有限公司        北向资金  8.185421e+10   
2     3                          中华人民共和国财政部          其他  2.782462e+10   
3     4                        中国证券金融股份有限公司        证金公司  7.941165e+09   
4     5                      中央汇金资产管理有限责任公司        汇金公司  1.810024e+09   
5     6                          香港中央结算有限公司        北向资金  7.691182e+08   
6     7  中国人寿保险股份有限公司-传统-普通保险产品-005L-CT001沪          保险  6.400483e+08   
7     8                     MUFG Bank, Ltd.          其他  5.203572e+08   
8     9  新华人寿保险股份有限公司-传统-普通保险产品-018L-CT001沪          保险  1.670908e+08   
9    10    太平人寿保险有限公司-传统-普通保险产品-022L-CT001沪          保险  1.386407e+08   

   hold_ratio_total change_status change_ratio  
0             58.59            不变         None  
1             25.40            减持         None  
2              8.64            不变         None  


## stock_name为"None"

In [28]:
raw_df = fetch_top_holders_em("000001", "20250331", True)

print(raw_df.columns.tolist())
display(raw_df.head())

['名次', '股东名称', '股东性质', '股份类型', '持股数', '占总流通股本持股比例', '增减', '变动比率']


,名次,股东名称,股东性质,股份类型,持股数,占总流通股本持股比例,增减,变动比率
0,1,中国平安保险(集团)股份有限公司-集团本级-自有资金,保险公司,A股,9618540236,49.565869,不变,NaN
1,2,中国平安人寿保险股份有限公司-自有资金,保险公司,A股,1186100488,6.112165,不变,NaN
2,3,香港中央结算有限公司,其它,A股,658114653,3.391370,-88767070,-11.885024
3,4,中国平安人寿保险股份有限公司-传统-普通保险产品,保险产品,A股,440478714,2.269857,不变,NaN
4,5,中国证券金融股份有限公司,证券公司,A股,429232688,2.211904,不变,NaN


In [13]:
stocks = pd.read_sql_query(
    "SELECT * FROM stocks ORDER BY stock_code",
    connection,
)

stock_000001 = stocks[stocks["stock_code"] == "000001"]

print(f"stocks 表记录数：{len(stocks)}")
display(stock_000001)

stocks 表记录数：813


,stock_code,stock_name,total_shares,float_shares,industry,updated_at
0,000001,平安银行,None,None,None,2026-08-20 06:09:09


In [25]:
# 查看 index_components 表结构
index_components_schema = pd.read_sql_query(
    "PRAGMA table_info(index_components)",
    connection,
)
display(index_components_schema)

# 查看 index_components 数据量
component_stats = pd.read_sql_query(
    """
    SELECT
        index_code,
        effective_date,
        COUNT(*) AS row_count,
        COUNT(DISTINCT stock_code) AS stock_count
    FROM index_components
    GROUP BY index_code, effective_date
    ORDER BY index_code, effective_date
    """,
    connection,
)
display(component_stats)

# 查看指数成分股明细，并关联指数基本信息
component_details = pd.read_sql_query(
    """
    SELECT
        ic.index_code,
        i.index_name,
        i.exchange,
        ic.stock_code,
        ic.stock_name,
        ic.weight,
        ic.effective_date
    FROM index_components AS ic
    LEFT JOIN indices AS i
        ON i.index_code = ic.index_code
    ORDER BY ic.index_code, ic.effective_date, ic.stock_code
    """,
    connection,
)

print(f"成分股记录数：{len(component_details)}")
display(component_details.head(50))

# 随机查看每个指数的成分股
random_component_samples = pd.concat(
    [
        group.sample(n=min(10, len(group)), random_state=42)
        for _, group in component_details.groupby("index_code")
    ],
    ignore_index=True,
).sort_values(["index_code", "stock_code"])

display(random_component_samples)

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,index_code,TEXT,1,None,0
2,2,stock_code,TEXT,1,None,0
3,3,stock_name,TEXT,0,None,0
4,4,weight,REAL,0,None,0
5,5,effective_date,DATE,0,None,0


,index_code,effective_date,row_count,stock_count
0,000300,2026-08-23,300,300
1,000688,2026-08-23,50,50
2,000905,2026-08-23,500,500
3,399006,2026-08-23,100,100


成分股记录数：950


,index_code,index_name,exchange,stock_code,stock_name,weight,effective_date
0,000300,沪深300,sh,000001,平安银行,0.433,2026-08-23
1,000300,沪深300,sh,000002,万科A,0.087,2026-08-23
2,000300,沪深300,sh,000063,中兴通讯,0.418,2026-08-23
3,000300,沪深300,sh,000100,TCL科技,0.378,2026-08-23
4,000300,沪深300,sh,000157,中联重科,0.145,2026-08-23
5,000300,沪深300,sh,000166,申万宏源,0.162,2026-08-23
6,000300,沪深300,sh,000301,东方盛虹,0.119,2026-08-23
7,000300,沪深300,sh,000333,美的集团,1.634,2026-08-23
8,000300,沪深300,sh,000338,潍柴动力,0.577,2026-08-23
9,000300,沪深300,sh,000408,藏格矿业,0.243,2026-08-23


,index_code,index_name,exchange,stock_code,stock_name,weight,effective_date
8,000300,沪深300,sh,000166,申万宏源,0.162,2026-08-23
3,000300,沪深300,sh,000408,藏格矿业,0.243,2026-08-23
7,000300,沪深300,sh,301308,江波龙,0.338,2026-08-23
2,000300,沪深300,sh,600372,中航机载,0.105,2026-08-23
9,000300,沪深300,sh,600760,中航沈飞,0.189,2026-08-23
6,000300,沪深300,sh,601009,南京银行,0.276,2026-08-23
0,000300,沪深300,sh,601077,渝农商行,0.138,2026-08-23
5,000300,沪深300,sh,601398,工商银行,0.992,2026-08-23
4,000300,沪深300,sh,601633,长城汽车,0.080,2026-08-23
1,000300,沪深300,sh,603019,中科曙光,0.469,2026-08-23


## 数据库回填

In [45]:
pd.read_sql_query(
    """
    SELECT COUNT(*) AS empty_count
    FROM top_holders
    WHERE stock_name IS NULL OR stock_name = ''
    """,
    connection,
)
# 待修复记录

,empty_count
0,0


In [26]:
stocks = pd.read_sql_query(
    """
    SELECT stock_code, stock_name
    FROM stocks
    ORDER BY stock_code
    """,
    connection,
)

print(f"stocks 记录数：{len(stocks)}")
display(stocks.head())
# 确认stocks 表中有多少数据

stocks 记录数：1694


,stock_code,stock_name
0,000001,平安银行
1,000002,万科A
2,000009,中国宝安
3,000020,深华发Ａ
4,000021,深科技


In [44]:
connection.execute(
    """
    UPDATE top_holders
    SET stock_name = (
        SELECT s.stock_name
        FROM stocks AS s
        WHERE s.stock_code = top_holders.stock_code
    )
    WHERE (top_holders.stock_name IS NULL OR top_holders.stock_name = '')
      AND EXISTS (
          SELECT 1
          FROM stocks AS s
          WHERE s.stock_code = top_holders.stock_code
            AND s.stock_name IS NOT NULL
            AND s.stock_name != ''
      )
    """
)

connection.commit()
# 回填操作

## 最近的报告期

In [27]:
report_periods = pd.read_sql_query(
    """
    SELECT
        report_date,
        COUNT(*) AS row_count,
        COUNT(DISTINCT stock_code) AS stock_count
    FROM top_holders
    GROUP BY report_date
    ORDER BY report_date DESC
    """,
    connection,
)

display(report_periods)

target_date = "20260630"
latest_date = report_periods["report_date"].iloc[0] if not report_periods.empty else None

print(f"数据库中最新报告期：{latest_date}")
print(f"目标报告期：{target_date}")

try:
    target_df = fetch_top_holders_em("000001", target_date, True)
    print(f"{target_date} 获取成功，记录数：{len(target_df)}")
    display(target_df.head())
except Exception as error:
    print(f"{target_date} 暂时无法获取：{error}")

,report_date,row_count,stock_count
0,2026-06-30,23185,1159
1,2026-03-31,33869,1693
2,2025-12-31,33790,1690
3,2025-09-30,33561,1675
4,2025-06-30,33518,1674
5,2025-03-31,33392,1664
6,2024-12-31,33406,1668


数据库中最新报告期：2026-06-30
目标报告期：20260630
20260630 暂时无法获取：name 'fetch_top_holders_em' is not defined


# 机构调研

In [28]:
research_tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
      AND (
          name LIKE '%research%'
          OR name LIKE '%institution%'
          OR name LIKE '%调研%'
      )
    ORDER BY name
    """,
    connection,
)

display(research_tables)

if not research_tables.empty:
    research_table = research_tables.iloc[0]["name"]
    research_count = connection.execute(
        f"SELECT COUNT(*) FROM [{research_table}]"
    ).fetchone()[0]

    print(f"机构调研数据表：{research_table}")
    print(f"记录数：{research_count}")

    research_data = pd.read_sql_query(
        f"SELECT * FROM [{research_table}] LIMIT 20",
        connection,
    )
    display(research_data)
else:
    print("未找到机构调研数据表")

,name
0,institutional_holdings
1,institutional_research


机构调研数据表：institutional_holdings
记录数：5008


,stock_code,stock_name,holder_types,report_date,last_scan_date,created_at
0,000001,平安银行,"保险,北向资金",2026-06-30,2026-08-23,2026-08-23 04:42:55
1,000002,万 科Ａ,NaN,NaN,2026-08-23,2026-08-23 04:42:55
2,000006,深振业Ａ,NaN,NaN,2026-08-23,2026-08-23 04:42:56
3,000007,全新好,NaN,NaN,2026-08-23,2026-08-23 04:42:56
4,000008,神州高铁,NaN,NaN,2026-08-23,2026-08-23 04:42:56
5,000009,中国宝安,NaN,NaN,2026-08-23,2026-08-23 04:42:56
6,000011,深物业A,NaN,NaN,2026-08-23,2026-08-23 04:42:56
7,000012,南 玻Ａ,NaN,NaN,2026-08-23,2026-08-23 04:42:57
8,000014,沙河股份,NaN,NaN,2026-08-23,2026-08-23 04:42:57
9,000017,深中华A,NaN,NaN,2026-08-23,2026-08-23 04:42:57


In [7]:
import akshare as ak

df = ak.stock_hsgt_individual_em(symbol="000001")
print(df["持股日期"].min(), df["持股日期"].max())

2017-03-16 2024-08-16


## 全市场机构调研与非成分股候选

默认 `research` 保存全市场调研记录；`--research-full-market` 用于补齐历史日期中此前遗漏的非成分股记录。以下查询统计最近 30 天的非指数成分股调研活跃度。

In [29]:
from analysis.research_candidates import get_research_candidates

research_coverage = pd.read_sql_query(
    """
    SELECT MIN(survey_date) AS min_date,
           MAX(survey_date) AS max_date,
           COUNT(*) AS row_count,
           COUNT(DISTINCT stock_code) AS stock_count
    FROM institutional_research
    """,
    connection,
)

print("机构调研覆盖情况：")
display(research_coverage)

candidates = get_research_candidates(days=30)
print(f"近30天非成分股候选数量：{len(candidates)}")
display(candidates.head(50))

机构调研覆盖情况：


,min_date,max_date,row_count,stock_count
0,2026-01-01,2026-08-21,123020,4188


近30天非成分股候选数量：258


,stock_code,stock_name,survey_count,institution_count,quality_institution_count,latest_survey_date,score
0,688167,炬光科技,323,323,94,2026-08-12,108.452
1,688005,容百科技,139,139,54,2026-07-30,66.354
2,002706,良信股份,116,116,51,2026-08-17,62.905
3,688112,鼎阳科技,104,104,48,2026-08-17,59.635
4,600096,云天化,147,140,46,2026-08-18,58.420
5,688083,中望软件,113,113,43,2026-08-18,54.840
6,300818,耐普矿机,108,108,43,2026-08-17,54.728
7,002886,沃特股份,56,56,40,2026-08-18,50.108
8,300693,盛弘股份,149,135,35,2026-08-17,47.380
9,002563,森马服饰,61,61,35,2026-08-18,45.318


# 行情数据

In [31]:
# 查看数据库中可能与行情相关的数据表及记录数
market_tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
      AND (
          name LIKE '%price%'
          OR name LIKE '%quote%'
          OR name LIKE '%market%'
          OR name LIKE '%daily%'
          OR name LIKE '%行情%'
      )
    ORDER BY name
    """,
    connection,
)

display(market_tables)

for table_name in market_tables["name"]:
    count = connection.execute(
        f"SELECT COUNT(*) FROM [{table_name}]"
    ).fetchone()[0]

    print(f"\n表名：{table_name}，记录数：{count}")
    display(pd.read_sql_query(
        f"SELECT * FROM [{table_name}] LIMIT 10",
        connection,
    ))

,name
0,daily_prices



表名：daily_prices，记录数：701213


,id,stock_code,trade_date,close_price,open_price,high_price,low_price,volume,amount
0,53988,689009,2026-05-25,37.73,38.48,38.66,37.66,98659.0,386318456.0
1,53989,689009,2026-05-26,38.29,37.79,38.41,37.58,89760.0,351839340.0
2,53990,689009,2026-05-27,36.78,38.27,38.46,36.73,114027.0,439048568.0
3,53991,689009,2026-05-28,37.70,37.33,38.22,36.56,142372.0,550760818.0
4,53992,689009,2026-05-29,37.20,37.44,37.97,36.78,109676.0,423460521.0
5,53993,689009,2026-06-01,37.45,37.08,38.23,36.80,112110.0,432539234.0
6,53994,689009,2026-06-02,36.62,36.94,37.44,36.49,84657.0,322053957.0
7,53995,689009,2026-06-03,35.68,36.54,36.72,35.45,122190.0,453481800.0
8,53996,689009,2026-06-04,35.01,35.52,35.85,34.87,81435.0,296346114.0
9,53997,689009,2026-06-05,35.14,34.98,36.40,34.52,123092.0,449518098.0


## 行情接口测试

In [3]:
import akshare as ak
df = ak.stock_zh_a_daily(symbol='sz000001', start_date='20260801', end_date='20260822', adjust='qfq')
print('stock_zh_a_daily:', df.shape)
print(df.tail(3))

stock_zh_a_daily: (15, 9)
          date   open   high    low  close       volume        amount  \
12  2026-08-19  11.08  11.27  11.07  11.27  142649557.0  1.596599e+09   
13  2026-08-20  11.20  11.40  11.19  11.40  118357823.0  1.338933e+09   
14  2026-08-21  11.36  11.46  11.32  11.41   86912763.0  9.901121e+08   

    outstanding_share  turnover  
12       1.940568e+10  0.007351  
13       1.940568e+10  0.006099  
14       1.940568e+10  0.004479  


In [ ]:
stock_zh_a_hist

In [2]:
import akshare as ak

df = ak.stock_zh_a_hist(
    symbol="000001",
    period="daily",
    start_date="20260801",
    end_date="20260822",
    adjust="qfq",
)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df.head())

if not df.empty:
    print(f"\n最新收盘价: {df.iloc[-1]['收盘']}")

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))